# Data compilation script
Produces aggregated results over multiple training seeds.

Produces: `all_channels_results_qc_clean.csv`

In [1]:
import pandas as pd
from scipy.stats import pearsonr
import numpy as np
from functools import reduce
from scipy.stats import combine_pvalues
from statsmodels.stats.multitest import multipletests
import scanpy as sc
import anndata
from src.eval import calculate_results_per_batch
import torch

In [25]:
# Parameters
## Inputs
whitelist_df = pd.read_csv(f'../input_files/cid_whitelist.csv')
toremove_df = pd.read_csv(f'../input_files/cid_nuc_toremove.csv')
whitelist_ids = np.array(whitelist_df.columns)
remove_cell_ids = np.array(toremove_df.columns)
seeds = [0, 1, 2, 3, 4]
model_specs = { # Will be looked for in "out_dir/{model_path}/seed_X/test_preds.h5ad"
    "fucci405": {
        "path": f'fucci_final_nucmask_405-bg-random_resnet',
        "channel": "Hoechst",
    },
    "fuccibf": {
        "path": f'fucci_final_nucmask_bf-bg-random_resnet',
        "channel": "bf",
    },
    "fuccicellshape": {
        "path": f'fucci_final_nucmask_seg-bg-random_resnet',
        "channel": "Segmentation",
    },
    "fuccinucshape": {
        "path": f'fucci_final_nucmask_nucmask-bg-random_resnet',
        "channel": "NucleusSegmentation",
    },
    "fucci_angle": {
        "path": f'fucci_final_nucmask_488-561-seg-bg-random/mlp_fucci_angle_model',
        "channel": "FUCCI angle",
    },
}

## Outputs
out_dir = './results_rev_p01'
fig_dir = f'{out_dir}/Figures'
output_single_qc = f'test_results_qc.csv' # Will be created for each seed, for each model in "out_dir/{model_path}/seed_X/"
output_aggregated_qc = f'{out_dir}/all_channels_results_qc_clean.csv'

#### Aggregate results

In [26]:
for spec in model_specs.values():
    print(spec['path'])
    for seed in seeds:
        path = out_dir + '/' + spec['path'] + '/seed_{}/test_preds.h5ad'
        test_adata = anndata.read_h5ad(path.format(seed))
        keep_idxs = np.intersect1d(test_adata.obs.index, whitelist_ids)
        test_adata_ = test_adata[keep_idxs]
        print(test_adata_.shape, test_adata.shape)
        results_df = calculate_results_per_batch(test_adata_, test_adata_.layers['pred'], test_adata_.X)
        results_df.to_csv(out_dir + '/' + spec['path'] + '/seed_{}/'.format(seed) + output_single_qc)

fucci_final_nucmask_405-bg-random_resnet
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)
fucci_final_nucmask_bf-bg-random_resnet
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)
fucci_final_nucmask_seg-bg-random_resnet
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)
fucci_final_nucmask_nucmask-bg-random_resnet
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)
fucci_final_nucmask_488-561-seg-bg-random/mlp_fucci_angle_model
(1042, 2000) (1165, 2000)
(1049, 2000) (1165, 2000)
(1054, 2000) (1165, 2000)
(1055, 2000) (1165, 2000)
(1029, 2000) (1165, 2000)


In [24]:
results_df[results_df['genes'] == 'Top2a']


,genes,R2_JP277,Pearson_JP277,pvals_JP277,Spearman_JP277,pvals (spearman)_JP277,nonzero_Pearson_JP277,nonzero_pvals_JP277,R2_NG015,Pearson_NG015,...,R2_JP241,Pearson_JP241,pvals_JP241,Spearman_JP241,pvals (spearman)_JP241,nonzero_Pearson_JP241,nonzero_pvals_JP241,avg_Pearson,avg_Spearman,avg_R2
0,Top2a,0.143085,0.413694,7.769229e-13,0.37739,9.021706e-11,0.33284,8.415051e-08,0.010224,0.23238,...,-0.037011,0.156109,0.039114,0.191888,0.010961,0.0914,0.281074,0.232717,0.240662,-0.241574


In [27]:
def calculate_consensus(path_template, seeds, alpha=0.05, metric='avg_Pearson'):
    results_dfs = []
    pvals_dfs = []
    for seed in seeds:
        print(seed)
        # print(path_template.format(seed=seed))
        results_df = pd.read_csv(path_template.format(seed), index_col='genes')
        results_df[f'{metric}_{seed}'] = results_df[metric]

        # Combine p-values across batches
        test_batches = [s.split('_')[1] for s in results_df.columns if s.startswith('Pearson_')]
        batch_pval_columns = [f'pvals_{s}' for s in test_batches]
        pvalues = []
        for g, row in results_df[batch_pval_columns].iterrows():
            combined_pval = combine_pvalues(row.values, method='fisher')[1]
            
            pvalues.append(combined_pval)
        results_df[f'{metric}_pval'] = pvalues
        pvals_dfs.append(results_df[f'{metric}_pval'])
        results_dfs.append(results_df[f'{metric}_{seed}'])
    
    # Average results
    merged_df = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True), results_dfs)
    merged_df[metric] = merged_df.values.mean(axis=-1)

    # Combine p-values across seeds
    genes = merged_df.index
    combined_pvals_df = pd.DataFrame(index=genes, columns=['combined_pval'], dtype=float)

    for g in genes:
        pvals = [df.loc[g] for df in pvals_dfs]
        #combined_pvals_df.loc[g] = combine_pvalues(pvals, method='fisher')[1]
        combined_pvals_df.loc[g] = np.median(pvals)

    merged_df[f'{metric}_pval'] = combined_pvals_df[f'combined_pval']
    merged_df[f'{metric}_fdr'] =  multipletests(combined_pvals_df[f'combined_pval'], alpha=alpha, method='fdr_bh')[1]
    # false_discovery_control(combined_pvals_df[f'combined_pval'])
    return merged_df

In [28]:
dfs = []
for spec in model_specs.values():
    full_path = f"{out_dir}/{spec['path']}/seed_{{}}/{output_single_qc}"
    df = calculate_consensus(full_path, seeds, metric="avg_Pearson")
    df["channel"] = spec["channel"]
    dfs.append(df)

all_merged_df = pd.concat(dfs, axis=0)
all_merged_df = all_merged_df[['avg_Pearson', 'avg_Pearson_pval', 'avg_Pearson_fdr', 'channel']]

0
1
2
3
4
0
1
2
3
4
0
1
2
3
4
0
1
2
3
4
0
1
2
3
4


In [29]:
df_wide = all_merged_df.reset_index().pivot(index='genes', columns='channel', values=['avg_Pearson', 'avg_Pearson_pval', 'avg_Pearson_fdr'])
gene_mask = df_wide['avg_Pearson'].isna().any(axis=1)
df_wide = df_wide.loc[~gene_mask]
df_wide.to_csv(output_aggregated_qc)
df_wide[()]

avg_Pearson                                             \
channel       FUCCI angle   Hoechst NucleusSegmentation Segmentation   
genes                                                                  
1110032A03Rik    0.004182  0.005216            0.005969     0.020727   
1190007I07Rik    0.031311  0.050314            0.016058     0.031306   
1500011K16Rik    0.039375  0.079733            0.096026     0.062432   
1600010M07Rik    0.068624  0.093803            0.092678     0.084569   
1600019K03Rik    0.085950  0.151163            0.114040     0.189397   
...                   ...       ...                 ...          ...   
mt-Atp8          0.092526  0.085366            0.125996     0.136217   
mt-Nd6           0.094416  0.110767            0.056947     0.117452   
mt-Rnr1          0.143055  0.075974            0.061916     0.138809   
mt-Rnr2          0.190295  0.135564            0.124041     0.213389   
mt-Tm            0.010566  0.012299           -0.002209     0.026123   

                        avg_Pearson_pval                                \
channel              bf      FUCCI angle   Hoechst NucleusSegmentation   
genes                                                                    
1110032A03Rik  0.028213         0.659715  0.694888        4.417518e-01   
1190007I07Rik  0.035025         0.836240  0.227391        4.970666e-01   
1500011K16Rik  0.069997         0.450414  0.045834        1.509625e-02   
1600010M07Rik  0.101030         0.152829  0.012573        6.946435e-03   
1600019K03Rik  0.150083         0.056812  0.000027        2.870502e-03   
...                 ...              ...       ...                 ...   
mt-Atp8        0.140068         0.000325  0.000471        1.112233e-07   
mt-Nd6         0.096322         0.017538  0.000517        6.003178e-03   
mt-Rnr1        0.088771         0.002398  0.000354        2.295517e-05   
mt-Rnr2        0.173002         0.000001  0.000007        7.212995e-05   
mt-Tm          0.025981         0.522212  0.711966        6.188891e-01   

                                          avg_Pearson_fdr            \
channel        Segmentation            bf     FUCCI angle   Hoechst   
genes                                                                 
1110032A03Rik  5.546290e-01  8.926981e-01        0.717305  0.743951   
1190007I07Rik  8.744304e-01  5.341216e-01        0.844106  0.529566   
1500011K16Rik  1.212909e-01  9.627242e-02        0.622364  0.179098   
1600010M07Rik  1.905196e-02  2.064960e-03        0.412371  0.061976   
1600019K03Rik  1.834138e-08  7.857400e-05        0.210650  0.000238   
...                     ...           ...             ...       ...   
mt-Atp8        6.273960e-09  3.307588e-06        0.002530  0.003189   
mt-Nd6         4.750239e-04  1.243769e-03        0.087003  0.003469   
mt-Rnr1        6.362055e-06  2.642599e-07        0.015561  0.002506   
mt-Rnr2        7.391023e-11  1.645812e-09        0.000011  0.000067   
mt-Tm          3.183665e-01  2.588498e-01        0.657860  0.752419   

                                                               
channel       NucleusSegmentation  Segmentation            bf  
genes                                                          
1110032A03Rik            0.710974  6.816662e-01  8.946220e-01  
1190007I07Rik            0.716356  8.794740e-01  6.901327e-01  
1500011K16Rik            0.123154  3.819432e-01  3.212919e-01  
1600010M07Rik            0.072315  9.221788e-02  1.274610e-02  
1600019K03Rik            0.033935  3.762679e-07  7.026329e-04  
...                           ...           ...           ...  
mt-Atp8                  0.000007  1.367527e-07  4.156833e-05  
mt-Nd6                   0.063443  3.562679e-03  8.236695e-03  
mt-Rnr1                  0.000667  7.785147e-05  4.286541e-06  
mt-Rnr2                  0.001548  2.291217e-09  4.591814e-08  
mt-Tm                    0.744268  5.953368e-01  5.451609e-01  

[1395 rows x 15 columns]

In [20]:
df_wide[df_wide.index == 'Mustn1']

avg_Pearson                                            \
channel     Hoechst NucleusSegmentation Segmentation       bf   
genes                                                           
Mustn1     0.128573            0.068662     0.141949  0.15148   

        avg_Pearson_pval                                                  \
channel          Hoechst NucleusSegmentation  Segmentation            bf   
genes                                                                      
Mustn1      3.600763e-10            0.000095  6.734348e-11  1.996953e-10   

        avg_Pearson_fdr                                                  
channel         Hoechst NucleusSegmentation  Segmentation            bf  
genes                                                                    
Mustn1     8.371774e-09            0.001891  2.145855e-09  6.331248e-09